# Figure 5: Harmonization / Metric Generalizability

Builds 2×3 layout from data (standalone). Outputs: individual panels in `figures/Figure5/panels` + `Figure5_uniform_preview`.



## Shared Setup

Libraries, config, plot style, and data prep: age-effect outputs, metric–metric correlations, vendor-pair cross-correspondence, and shared theme/colors.


In [7]:
# --- Libraries ---
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(purrr)
  library(tibble)
  library(ggplot2)
  library(patchwork)
  library(cowplot)
  library(fs)
  library(jsonlite)
  library(scales)
  library(grid)
})

# --- Config and plot style ---
config_candidates <- c(
  Sys.getenv("CONFIG_PATH", unset = ""),
  fs::path(".", "config.json"),
  fs::path("..", "config.json"),
  fs::path("..", "..", "config.json")
)
config_candidates <- normalizePath(unique(config_candidates[nzchar(config_candidates)]), winslash = "/", mustWork = FALSE)
config_path <- config_candidates[file.exists(config_candidates)][1]
if (is.na(config_path) || !nzchar(config_path)) stop("Could not locate config.json")
config <- jsonlite::fromJSON(config_path)
project_root <- normalizePath(config$project_root, winslash = "/", mustWork = FALSE)
plot_style_file <- fs::path(project_root, "scripts", "utils", "plot_style.R")
if (!file.exists(plot_style_file)) stop("Missing plot style: ", plot_style_file)
source(plot_style_file)
font_family_use <- get_export_font_family()

# --- Microstructural metrics and vendor-pair definitions (metrics_keep, metric_labels, metric_order from plot_style.R) ---
# category_label_map, category_order_pretty, bundle_color_pretty from plot_style.R
pair_defs <- tibble::tribble(~pair_label, ~vendor_x, ~vendor_y, ~line_type, "GE-Philips", "GE", "Philips", "dotdash", "Siemens-GE", "Siemens", "GE", "solid", "Siemens-Philips", "Siemens", "Philips", "dotted")
age_effect_file <- fs::path(project_root, "data", "age_effects", "age_effects_all_outputs.rds")
if (!file.exists(age_effect_file)) stop("Missing assembled age-effect file: ", age_effect_file)
df_age_all <- readRDS(age_effect_file)

# --- Pooled age effects (harmonized; from scripts/4_age_effects) and metric–metric correlations ---
# Use no_quality as QC covariate for age effects
required_age_cols <- c("bundle", "bundle_category", "metric", "qc_metric", "source", "output_type", "scanner_manufacturer", "age_effect_size")
if (!all(required_age_cols %in% names(df_age_all))) stop("Assembled age-effect data missing required columns: ", paste(setdiff(required_age_cols, names(df_age_all)), collapse = ", "))
df_age <- df_age_all %>% filter(output_type == "pooled", qc_metric == "no_quality", scanner_manufacturer == "all", source == "harmonized", metric %in% metrics_keep, !is.na(bundle), !is.na(bundle_category), !is.na(age_effect_size)) %>% transmute(bundle = bundle, bundle_category_pretty = recode(bundle_category, !!!category_label_map), metric_label = unname(metric_labels[metric]), age_effect_size = as.numeric(age_effect_size))
if (nrow(df_age) == 0) stop("No rows for Figure 5 pooled (age_effects pooled + harmonized).")
wide_effects <- df_age %>% mutate(metric_label = factor(metric_label, levels = metric_order), bundle_category_pretty = factor(bundle_category_pretty, levels = category_order_pretty)) %>% distinct(bundle, bundle_category_pretty, metric_label, age_effect_size) %>% tidyr::pivot_wider(names_from = metric_label, values_from = age_effect_size) %>% tidyr::drop_na(all_of(metric_order))
if (nrow(wide_effects) < 10) stop("Too few complete bundles for Figure 5.")
pair_stats <- tidyr::expand_grid(metric_y = metric_order, metric_x = metric_order) %>% mutate(idx_y = match(metric_y, metric_order), idx_x = match(metric_x, metric_order), n = purrr::map2_int(metric_y, metric_x, function(my, mx) { d <- wide_effects %>% select(all_of(c(my, mx))) %>% tidyr::drop_na(); nrow(d) }), pearson_r = purrr::map2_dbl(metric_y, metric_x, function(my, mx) { if (identical(my, mx)) return(NA_real_); d <- wide_effects %>% select(all_of(c(my, mx))) %>% tidyr::drop_na(); if (nrow(d) < 3) return(NA_real_); cor(d[[my]], d[[mx]], method = "pearson") }), spearman_rho = purrr::map2_dbl(metric_y, metric_x, function(my, mx) { if (identical(my, mx)) return(NA_real_); d <- wide_effects %>% select(all_of(c(my, mx))) %>% tidyr::drop_na(); if (nrow(d) < 3) return(NA_real_); cor(d[[my]], d[[mx]], method = "spearman") }), r2 = pearson_r^2, show_cell = idx_y > idx_x, rho_plot = if_else(show_cell, spearman_rho, NA_real_), label = if_else(show_cell, sprintf("%.2f", rho_plot), NA_character_))
pair_unique <- pair_stats %>% filter(show_cell, !is.na(rho_plot)) %>% arrange(rho_plot)
worst_pair <- pair_unique %>% slice(1)
best_pair <- pair_unique %>% slice(n())
highlight_cells <- bind_rows(worst_pair %>% mutate(which_pair = "Worst"), best_pair %>% mutate(which_pair = "Best")) %>% distinct(metric_x, metric_y, which_pair)
x_plot_levels <- metric_order[-length(metric_order)]
y_plot_levels <- rev(metric_order[-1])
pair_stats_plot <- pair_stats %>% filter(metric_x %in% x_plot_levels, metric_y %in% y_plot_levels)
highlight_cells_plot <- highlight_cells %>% filter(metric_x %in% x_plot_levels, metric_y %in% y_plot_levels)
build_pair_df <- function(pair_row) { x_metric <- as.character(pair_row$metric_x[1]); y_metric <- as.character(pair_row$metric_y[1]); wide_effects %>% transmute(bundle = bundle, bundle_category_pretty = factor(bundle_category_pretty, levels = category_order_pretty), x_effect = .data[[x_metric]], y_effect = .data[[y_metric]]) %>% tidyr::drop_na(x_effect, y_effect) }
df_pair_worst <- build_pair_df(worst_pair)
df_pair_best <- build_pair_df(best_pair)

# --- Vendor-wise age effects: cross-vendor Spearman ρ of bundle-wise age effect vectors ---
df_age_vendor <- df_age_all %>% filter(output_type == "vendorwise", qc_metric == "no_quality", source %in% c("raw", "harmonized"), scanner_manufacturer %in% c("GE", "Philips", "Siemens"), metric %in% metrics_keep, !is.na(bundle), !is.na(bundle_category), !is.na(age_effect_size)) %>% transmute(bundle = bundle, bundle_category_pretty = recode(bundle_category, !!!category_label_map), metric_label = unname(metric_labels[metric]), source_pretty = if_else(source == "raw", "Unharmonized", "Harmonized"), scanner_manufacturer = as.character(scanner_manufacturer), age_effect_size = as.numeric(age_effect_size))
if (nrow(df_age_vendor) == 0) stop("No rows for Figure 5 top row (age_effects vendorwise, unharmonized/harmonized).")
# One row per (source, metric, bundle, vendor) so each vendor has one age effect per bundle.
df_vendor_long <- df_age_vendor %>% mutate(metric_label = factor(metric_label, levels = metric_order), source_pretty = factor(source_pretty, levels = c("Unharmonized", "Harmonized")), bundle_category_pretty = factor(bundle_category_pretty, levels = category_order_pretty)) %>% group_by(source_pretty, metric_label, bundle, scanner_manufacturer) %>% summarise(age_effect_size = first(age_effect_size), bundle_category_pretty = first(bundle_category_pretty), .groups = "drop")
# Cross-vendor ρ: for each (source, metric, vendor pair), join on bundle and Spearman cor of the two vectors.
panel_a_df <- df_vendor_long %>% group_by(source_pretty, metric_label) %>% group_modify(function(grp, key) { dx_all <- grp %>% select(bundle, scanner_manufacturer, age_effect_size); pair_defs %>% mutate(rho = purrr::pmap_dbl(list(vendor_x, vendor_y), function(vx, vy) { dx <- dx_all %>% filter(scanner_manufacturer == vx) %>% select(bundle, age_effect_size); dy <- dx_all %>% filter(scanner_manufacturer == vy) %>% select(bundle, age_effect_size); j <- inner_join(dx, dy, by = "bundle"); if (nrow(j) < 3) return(NA_real_); cor(j$age_effect_size.x, j$age_effect_size.y, method = "spearman", use = "pairwise.complete.obs") }), n = purrr::pmap_int(list(vendor_x, vendor_y), function(vx, vy) { dx <- dx_all %>% filter(scanner_manufacturer == vx) %>% select(bundle); dy <- dx_all %>% filter(scanner_manufacturer == vy) %>% select(bundle); nrow(inner_join(dx, dy, by = "bundle")) })) }) %>% ungroup() %>% mutate(pair_label = factor(pair_label, levels = pair_defs$pair_label))
# Wide table for scatters: one row per (source, metric, bundle) with GE, Philips, Siemens columns.
wide_vendor <- df_vendor_long %>% tidyr::pivot_wider(names_from = scanner_manufacturer, values_from = age_effect_size)
gain_df <- panel_a_df %>% select(source_pretty, metric_label, pair_label, vendor_x, vendor_y, rho) %>% tidyr::pivot_wider(names_from = source_pretty, values_from = rho) %>% mutate(gain = Harmonized - Unharmonized) %>% arrange(desc(gain))
best_gain <- gain_df %>% slice(1)
selected_metric <- as.character(best_gain$metric_label[1])
selected_vendor_x <- as.character(best_gain$vendor_x[1])
selected_vendor_y <- as.character(best_gain$vendor_y[1])
selected_pair_label <- as.character(best_gain$pair_label[1])
scatter_df <- wide_vendor %>% filter(metric_label == selected_metric) %>% transmute(source_pretty = factor(source_pretty, levels = c("Unharmonized", "Harmonized")), bundle = bundle, bundle_category_pretty = factor(bundle_category_pretty, levels = category_order_pretty), x_effect = .data[[selected_vendor_x]], y_effect = .data[[selected_vendor_y]]) %>% tidyr::drop_na(x_effect, y_effect)
rho_by_source <- scatter_df %>% group_by(source_pretty) %>% summarise(rho = cor(x_effect, y_effect, method = "spearman"), .groups = "drop")

# --- Shared plot style (theme, colors, line/point sizes) ---
bg_col <- "#FFFFFF"
# Uniform style settings
title_pt <- 7
text_pt <- 6
axis_line_w <- 0.18
tick_line_w <- 0.18
tick_len_pt <- 2
line_w <- 0.32
point_sz <- 1.00

base_theme_sq <- function() {
  theme_minimal(base_size = text_pt, base_family = font_family_use) +
    theme(
      text = element_text(size = text_pt, family = font_family_use),
      plot.title = element_text(size = title_pt, face = "plain", hjust = 0.5),
      axis.title = element_text(size = text_pt),
      axis.text = element_text(size = text_pt),
      panel.grid = element_blank(),
      panel.background = element_rect(fill = bg_col, color = NA),
      plot.background = element_rect(fill = bg_col, color = NA),
      axis.line = element_line(color = "black", linewidth = axis_line_w),
      axis.ticks = element_line(color = "black", linewidth = tick_line_w),
      axis.ticks.length = unit(tick_len_pt, "pt"),
      plot.margin = margin(3, 3, 3, 3),
      aspect.ratio = 1
    )
}

# ---------- Top row (cross-vendor) and bottom row (metric-age) panels ----------
metric_levels <- levels(panel_a_df$metric_label)
fallback <- scales::hue_pal()(length(metric_levels))
names(fallback) <- metric_levels
metric_colors_panel <- if (is.null(metric_colors) || length(metric_colors) == 0 || is.null(names(metric_colors)) || any(!nzchar(names(metric_colors)))) {
  fallback
} else {
  mc <- metric_colors[metric_levels]
  mc[is.na(mc)] <- fallback[names(mc[is.na(mc)])]
  mc
}
pair_linetypes <- c("GE-Philips" = "dotdash", "Siemens-GE" = "solid", "Siemens-Philips" = "dotted")


## Panel A (top left): Cross-vendor age effect correspondence

Line plot of cross-vendor age effect Spearman ρ (Unharmonized vs Harmonized) for each microstructural metric and vendor pair.


In [8]:
p_top_left <- ggplot(
  panel_a_df,
  aes(x = source_pretty, y = rho, color = metric_label, linetype = pair_label, group = interaction(metric_label, pair_label))
) +
  geom_line(linewidth = line_w, show.legend = FALSE) +
  geom_point(size = point_sz, show.legend = FALSE) +
  scale_color_manual(values = metric_colors_panel, name = "Microstructural Metric", drop = FALSE) +
  scale_linetype_manual(values = pair_linetypes, name = "Vendor Pair", drop = FALSE) +
  scale_x_discrete(expand = expansion(mult = c(0.06, 0.06))) +
  scale_y_continuous(limits = c(0.4, 1.02), breaks = c(0.4, 0.6, 0.8, 1.0), expand = expansion(mult = c(0, 0))) +
  labs(
    title = "Cross-Vendor Age Effect Correspondence",
    x = NULL,
    y = expression(paste("Cross-Vendor Age Effect Correspondence (", rho, ")"))
  ) +
  base_theme_sq() +
  theme(legend.position = "none") +
  guides(color = "none", linetype = "none")


## Panel B (top row): Unharmonized and harmonized vendor-pair scatters

Scatter of age effects for the vendor pair with largest harmonization gain (selected metric). Left: unharmonized; right: harmonized.


In [9]:
pair_limits <- range(c(scatter_df$x_effect, scatter_df$y_effect), na.rm = TRUE)
pair_pad <- diff(pair_limits) * 0.04
pair_limits <- pair_limits + c(-pair_pad, pair_pad)
x_axis_label <- bquote(.(selected_vendor_x) ~ "Age Effect (" * Delta * R[adj]^2 * ")")
y_axis_label <- bquote(.(selected_vendor_y) ~ "Age Effect (" * Delta * R[adj]^2 * ")")
make_top_scatter <- function(source_level) {
  df <- scatter_df %>% filter(source_pretty == source_level)
  rho_val <- rho_by_source %>% filter(source_pretty == source_level) %>% pull(rho)
  ggplot(df, aes(x = x_effect, y = y_effect, color = bundle_category_pretty)) +
    geom_abline(intercept = 0, slope = 1, linetype = "dotted", color = "grey45", linewidth = line_w, show.legend = FALSE) +
    geom_point(size = point_sz, alpha = 0.85, show.legend = FALSE) +
    geom_smooth(aes(group = 1), method = "lm", se = TRUE, color = "black", fill = "grey80", linewidth = line_w, alpha = 0.35, show.legend = FALSE) +
    scale_color_manual(values = bundle_color_pretty, name = "Bundle Category", drop = FALSE) +
    labs(
      title = sprintf("%s %s %s", selected_pair_label, as.character(source_level), selected_metric),
      x = x_axis_label,
      y = y_axis_label
    ) +
    annotate(
      "text",
      x = quantile(df$x_effect, 0.08, na.rm = TRUE),
      y = max(df$y_effect, na.rm = TRUE),
      label = sprintf("rho == %.2f", rho_val),
      parse = TRUE,
      hjust = 0, vjust = 1,
      size = text_pt / .pt
    ) +
    coord_fixed(ratio = 1, xlim = pair_limits, ylim = pair_limits, clip = "off") +
    base_theme_sq() +
    theme(legend.position = "none") +
    guides(color = "none")
}
p_top_mid <- make_top_scatter("Unharmonized")
p_top_right <- make_top_scatter("Harmonized")


## Panel C (bottom left): Cross-metric age-effect triangle

Heatmap of Spearman ρ between metric-wise age effects across bundles. Worst and best metric pairs highlighted.


In [10]:
# ---------- Bottom row plot objects (use pair_stats_plot, highlight_cells_plot, etc. from above) ----------
matrix_df <- pair_stats_plot %>%
  filter(show_cell) %>%
  mutate(
    metric_x = factor(metric_x, levels = x_plot_levels),
    metric_y = factor(metric_y, levels = y_plot_levels)
  )
highlight_df <- highlight_cells_plot %>%
  mutate(
    metric_x = factor(metric_x, levels = x_plot_levels),
    metric_y = factor(metric_y, levels = y_plot_levels)
  )
p_bottom_left <- ggplot(matrix_df, aes(x = metric_x, y = metric_y, fill = rho_plot)) +
  geom_tile(color = "#F0F0F0", linewidth = 0.35) +
  geom_tile(data = highlight_df, fill = NA, color = "#56B4E9", linewidth = 1.1) +
  geom_text(aes(label = sprintf("%.2f", rho_plot)), size = text_pt / .pt, family = font_family_use) +
  scale_fill_gradient(
    low = "#FFFFFF", high = "#D7191C",
    limits = c(0.5, 1.0), breaks = c(0.5, 0.75, 1.0),
    oob = scales::squish,
    name = expression(rho)
  ) +
  labs(title = "Cross-Metric Age Effect Correspondence", x = NULL, y = NULL) +
  coord_fixed(clip = "off") +
  base_theme_sq() +
  theme(
    legend.position = c(0.90, 0.52),
    legend.background = element_blank(),
    legend.title = element_text(size = text_pt),
    legend.text = element_text(size = text_pt),
    legend.key.height = unit(7, "pt"),
    legend.key.width = unit(8, "pt")
  )


## Panel D & E (bottom row): Worst and best metric-pair scatters

Scatter of bundle age effects for the metric pair with lowest (worst) and highest (best) cross-metric agreement.


In [11]:
make_bottom_scatter <- function(df, x_lab_metric, y_lab_metric, title_text) {
  rho_val <- cor(df$x_effect, df$y_effect, method = "spearman")
  lims <- range(c(df$x_effect, df$y_effect), na.rm = TRUE)
  pad <- diff(lims) * 0.06
  lims <- lims + c(-pad, pad)
  ggplot(df, aes(x = x_effect, y = y_effect, color = bundle_category_pretty)) +
    geom_abline(intercept = 0, slope = 1, linetype = "dotted", color = "grey45", linewidth = line_w, show.legend = FALSE) +
    geom_point(size = point_sz, alpha = 0.85, show.legend = FALSE) +
    geom_smooth(aes(group = 1), method = "lm", se = TRUE, color = "black", fill = "grey80", linewidth = line_w, alpha = 0.35, show.legend = FALSE) +
    scale_color_manual(values = bundle_color_pretty, name = "Bundle Category", drop = FALSE) +
    labs(
      title = title_text,
      x = bquote(.(x_lab_metric) ~ "Age Effect (" * Delta * R[adj]^2 * ")"),
      y = bquote(.(y_lab_metric) ~ "Age Effect (" * Delta * R[adj]^2 * ")")
    ) +
    annotate(
      "text",
      x = quantile(df$x_effect, 0.08, na.rm = TRUE),
      y = max(df$y_effect, na.rm = TRUE),
      label = sprintf("rho == %.2f", rho_val),
      parse = TRUE,
      hjust = 0, vjust = 1,
      size = text_pt / .pt
    ) +
    coord_fixed(ratio = 1, xlim = lims, ylim = lims, clip = "off") +
    base_theme_sq() +
    theme(legend.position = "none") +
    guides(color = "none")
}
p_bottom_mid <- make_bottom_scatter(df_pair_worst, worst_pair$metric_x, worst_pair$metric_y, "Worst Pair Agreement")
p_bottom_right <- make_bottom_scatter(df_pair_best, best_pair$metric_x, best_pair$metric_y, "Best Pair Agreement")


## Assemble and export

Build shared legends, combine rows, save individual panels and combined preview (PNG + PDF).


In [12]:
# ---------- Shared legends ----------
legend_metric_df <- panel_a_df %>%
  distinct(metric_label, pair_label) %>%
  as.data.frame()
legend_metric_df <- merge(
  legend_metric_df,
  data.frame(dummy_x = factor(c("x1", "x2"), levels = c("x1", "x2")), dummy_y = 1),
  all = TRUE
)
legend_metric_plot <- ggplot(
  legend_metric_df,
  aes(x = dummy_x, y = dummy_y, color = metric_label, linetype = pair_label, group = interaction(metric_label, pair_label))
) +
  geom_line(linewidth = line_w) +
  geom_point(size = point_sz) +
  scale_color_manual(values = metric_colors_panel, name = "Microstructural Metric", drop = FALSE) +
  scale_linetype_manual(values = pair_linetypes, name = "Vendor Pair", drop = FALSE) +
  guides(
    color = guide_legend(order = 1, nrow = 1, byrow = TRUE),
    linetype = guide_legend(order = 2, nrow = 1, byrow = TRUE)
  ) +
  theme_void() +
  theme(
    text = element_text(family = font_family_use, size = text_pt),
    legend.position = "bottom",
    legend.direction = "horizontal",
    legend.box = "vertical",
    legend.title = element_text(face = "plain", size = text_pt),
    legend.text = element_text(size = text_pt),
    legend.key.height = unit(6, "pt"),
    legend.key.width = unit(10, "pt"),
    legend.spacing.x = unit(3, "pt"),
    legend.spacing.y = unit(1, "pt"),
    legend.box.spacing = unit(2, "pt")
  )
legend_bundle_df <- data.frame(
  bundle_category_pretty = factor(names(bundle_color_pretty), levels = names(bundle_color_pretty)),
  x = 1, y = 1
)
legend_bundle_plot <- ggplot(legend_bundle_df, aes(x = x, y = y, color = bundle_category_pretty)) +
  geom_point(size = point_sz + 0.2) +
  scale_color_manual(values = bundle_color_pretty, name = "Bundle Category", drop = FALSE,
    guide = guide_legend(nrow = 2, byrow = TRUE)) +
  theme_void() +
  theme(
    text = element_text(family = font_family_use, size = text_pt),
    legend.position = "bottom",
    legend.direction = "horizontal",
    legend.title = element_text(face = "plain", size = text_pt),
    legend.text = element_text(size = text_pt),
    legend.key.height = unit(6, "pt"),
    legend.key.width = unit(8, "pt"),
    legend.spacing.x = unit(3, "pt"),
    legend.spacing.y = unit(1, "pt"),
    legend.box.spacing = unit(1, "pt")
  )
p_metric_legend <- cowplot::ggdraw(cowplot::get_legend(legend_metric_plot))
p_bundle_legend <- cowplot::ggdraw(cowplot::get_legend(legend_bundle_plot))
legend_row <- p_metric_legend + p_bundle_legend + patchwork::plot_layout(widths = c(1, 1))
# ---------- Assemble ----------
row_top <- p_top_left + p_top_mid + p_top_right + patchwork::plot_layout(widths = c(1, 1, 1))
row_bottom <- p_bottom_left + p_bottom_mid + p_bottom_right + patchwork::plot_layout(widths = c(1, 1, 1))
combined_56_uniform <- row_top / legend_row / row_bottom + patchwork::plot_layout(heights = c(1, 0.22, 1))
figure_width_in <- 180 / 25.4
panel_side_in <- figure_width_in / 3
legend_height_in <- panel_side_in * 0.40
figure_height_in <- panel_side_in * 2 + legend_height_in
figure5_dir <- fs::path(project_root, "figures", "Figure5")
figure5_panels_dir <- fs::path(figure5_dir, "panels")
fs::dir_create(figure5_panels_dir, recurse = TRUE)
panel_in <- (180 / 25.4) / 3
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_a_cross_vendor_rho_lines.png"), p_top_left, width = panel_in, height = panel_in * 0.85, units = "in", dpi = 600, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_a_cross_vendor_rho_lines.pdf"), p_top_left, width = panel_in, height = panel_in * 0.85, units = "in", device = grDevices::cairo_pdf, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_raw_scatter.png"), p_top_mid, width = panel_in, height = panel_in, units = "in", dpi = 600, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_raw_scatter.pdf"), p_top_mid, width = panel_in, height = panel_in, units = "in", device = grDevices::cairo_pdf, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_harmonized_scatter.png"), p_top_right, width = panel_in, height = panel_in, units = "in", dpi = 600, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_harmonized_scatter.pdf"), p_top_right, width = panel_in, height = panel_in, units = "in", device = grDevices::cairo_pdf, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_a_metric_age_effect_rho_triangle.png"), p_bottom_left, width = panel_in, height = panel_in, units = "in", dpi = 600, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_a_metric_age_effect_rho_triangle.pdf"), p_bottom_left, width = panel_in, height = panel_in, units = "in", device = grDevices::cairo_pdf, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_worst_pair_scatter.png"), p_bottom_mid, width = panel_in, height = panel_in, units = "in", dpi = 600, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_worst_pair_scatter.pdf"), p_bottom_mid, width = panel_in, height = panel_in, units = "in", device = grDevices::cairo_pdf, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_best_pair_scatter.png"), p_bottom_right, width = panel_in, height = panel_in, units = "in", dpi = 600, bg = "white")
ggsave(fs::path(figure5_panels_dir, "Figure5_panel_b_best_pair_scatter.pdf"), p_bottom_right, width = panel_in, height = panel_in, units = "in", device = grDevices::cairo_pdf, bg = "white")
if (interactive()) {
  options(repr.plot.width = figure_width_in, repr.plot.height = figure_height_in, repr.plot.res = 120)
  print(combined_56_uniform)
}
ggsave(
  filename = fs::path(figure5_dir, "Figure5_uniform_preview.png"),
  plot = combined_56_uniform,
  width = figure_width_in,
  height = figure_height_in,
  units = "in",
  dpi = 600,
  bg = bg_col
)
tryCatch(
  ggsave(
    filename = fs::path(figure5_dir, "Figure5_uniform_preview.pdf"),
    plot = combined_56_uniform,
    width = figure_width_in,
    height = figure_height_in,
    units = "in",
    device = cairo_pdf,
    bg = bg_col
  ),
  error = function(e) {
    message("[WARN] cairo_pdf export failed (", conditionMessage(e), "). Falling back to base pdf device.")
    ggsave(
      filename = fs::path(figure5_dir, "Figure5_uniform_preview.pdf"),
      plot = combined_56_uniform,
      width = figure_width_in,
      height = figure_height_in,
      units = "in",
      device = grDevices::pdf,
      bg = bg_col
    )
  }
)
cat("Saved:\n",
    fs::path(figure5_dir, "Figure5_uniform_preview.png"), "\n",
    fs::path(figure5_dir, "Figure5_uniform_preview.pdf"), "\n",
    sep = "")


Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warning message in grid.Call(C_textBounds, as.graphicsAnnot(x$label), x$x, x$y, :
"font family 'Arial' not found in PostScript font database"
Warnin

Saved:
/Users/stevenmeisler/Documents/Penn/abcc-qsiprep/meisler_abcd_dmri/figures/Figure5/Figure5_uniform_preview.png
/Users/stevenmeisler/Documents/Penn/abcc-qsiprep/meisler_abcd_dmri/figures/Figure5/Figure5_uniform_preview.pdf
